# 服务运行与多进程部署

学习目标：用真实服务观察进程内状态与应用重启，理解多 worker 的管理方式，辨别开发重载的用途，解释进程内状态和资源的归属，并正确关闭服务。

前置知识：进程、网络端口、环境变量、应用生命周期与上下文管理器。

环境准备：[FastAPI 环境与运行说明](README.md)。本章使用 Python 3.12、FastAPI 和 Uvicorn 0.53.0；并发限制反例明确使用 h11 协议实现。

工作目录：content/Web与应用开发/FastAPI。先执行 Notebook 内的小例子，再按各节步骤在终端启动服务，运行紧随其后的 Code 单元；切换模式前关闭上一服务。实验使用本机 8150、8151 端口，最多同时运行两个服务进程，临时文件由上下文管理器关闭并删除。源码变更使用正常停止后重新启动；自动重载仅说明机制与当前环境限制。

配套脚本：位于 scripts/15-service-and-workers/。

1. [app.py](scripts/15-service-and-workers/app.py)：供 Uvicorn 导入的小应用，返回 PID、进程内计数和版本标记，并记录生命周期。

## 1 看清计数属于哪个进程

先写一个每次 POST 都加一的接口。os.getpid() 返回当前进程标识（PID）；app.state 保存这个应用实例的状态。路由通过 request.app 取得正在处理请求的应用。

In [1]:
import os

from fastapi import FastAPI, Request
from fastapi.testclient import TestClient


app = FastAPI()
app.state.count = 0


@app.post("/tick")
async def tick(request: Request):
    request.app.state.count += 1
    return {"pid": os.getpid(), "count": request.app.state.count}

C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


TestClient 在当前 Python 进程中调用应用。这里先确认同一份状态能连续累加；它还没有启动独立 Uvicorn worker。

In [2]:
with TestClient(app) as client:
    first = client.post("/tick").json()
    second = client.post("/tick").json()

assert first["pid"] == second["pid"] == os.getpid()
assert [first["count"], second["count"]] == [1, 2]
print(first, second)  # 预期：两个结果的 pid 相同，count 依次为 1、2；pid 的具体数字随进程变化。
# PID 属于当前内核进程；它不是后面真实服务的 worker PID。

{'pid': 35084, 'count': 1} {'pid': 35084, 'count': 2}


## 2 把初始化与资源关闭放在 lifespan

lifespan 的 yield 之前初始化状态，yield 之后完成关闭。Uvicorn 的每个 worker 都独立运行应用生命周期，所以“启动一次”是每个应用实例一次。

用一个小型临时文件表示应用持有的资源：启动时写入当前 PID，运行期间保持打开，退出 with 时关闭并删除。它位于系统临时目录，不保存业务数据；计数也在每次启动时归零。

In [3]:
from contextlib import asynccontextmanager
from tempfile import TemporaryFile


@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.count = 0
    journal = TemporaryFile(mode="w+t", encoding="utf-8")
    app.state.journal = journal
    try:
        with journal:
            journal.write(f"pid={os.getpid()}\n")
            print(f"startup pid={os.getpid()}")  # 预期：进入 lifespan 时显示 startup 与当前 pid；仅定义函数时没有输出。
            yield
            print(f"shutdown pid={os.getpid()}")  # 预期：退出 lifespan 时显示 shutdown，与本次启动日志的 pid 相同。
    finally:
        print(f"closed={journal.closed}")  # 预期：离开文件上下文后显示 closed=True。

新应用在响应中增加 journal_closed，直接观察请求期间资源是否仍然打开。这里只进行少量顺序请求，计数用途是展示内存归属。

In [4]:
managed_app = FastAPI(lifespan=lifespan)


@managed_app.post("/tick")
async def managed_tick(request: Request):
    request.app.state.count += 1
    return {
        "pid": os.getpid(), "count": request.app.state.count,
        "journal_closed": request.app.state.journal.closed,
    }

将 TestClient 用作上下文管理器，进入和退出时分别执行应用的启动与关闭。正常退出会经过 yield 后的 shutdown 提示；若异常离开，with 仍关闭文件，外层 finally 记录实际 closed 状态。

In [5]:
with TestClient(managed_app) as client:
    result = client.post("/tick").json()
    print(result)  # 预期：启动日志之后显示 count=1、journal_closed=False 和本次 pid；退出上下文后再显示关闭日志。
    assert result["count"] == 1 and result["journal_closed"] is False
assert managed_app.state.journal.closed
# 退出时出现 shutdown，随后 closed=True；请求结束时资源还没有关闭。

startup pid=35084
{'pid': 35084, 'count': 1, 'journal_closed': False}
shutdown pid=35084
closed=True


## 3 启动单 worker 服务

配套 app.py 将前面的生命周期和计数整理成可导入模块，并增加 GET /state 查询状态与 LABEL 版本标记。该查询不会增加计数；POST /tick 才会。导入模块本身不启动服务器。

app:app 的冒号左边是模块 app，右边是其中的应用对象 app。--app-dir 指定模块所在目录，--host 与 --port 指定监听地址；127.0.0.1 只用于本机，0.0.0.0 表示绑定所有 IPv4 网络接口，客户端仍须访问实际地址。

Step 1：在课程目录的终端启动一个 worker。

```bash
python -m uvicorn app:app --app-dir scripts/15-service-and-workers --host 127.0.0.1 --port 8150 --workers 1 --log-level info --timeout-graceful-shutdown 5
```

Step 2：等待终端出现 Application startup complete，再执行下面的请求。

HTTPX 的 with 会关闭客户端连接，trust_env=False 避免本地请求使用环境代理；timeout=2 为网络等待设置超时，单位为秒，不是整段实验的总耗时上限。

In [6]:
import httpx


BASE_URL = "http://127.0.0.1:8150"
single_results = []
with httpx.Client(base_url=BASE_URL, timeout=2, trust_env=False) as client:
    for _ in range(3):
        response = client.post("/tick")
        response.raise_for_status()
        single_results.append(response.json())
assert len({item["pid"] for item in single_results}) == 1
assert [item["count"] for item in single_results] == [1, 2, 3]
print(single_results)  # 预期：三项 pid 相同，count 依次为 1、2、3，label='v1'，文件均未关闭。
# 从刚启动且未调用过 /tick 的服务开始，三次响应属于同一个 PID。

[{'pid': 31368, 'count': 1, 'label': 'v1', 'journal_closed': False}, {'pid': 31368, 'count': 2, 'label': 'v1', 'journal_closed': False}, {'pid': 31368, 'count': 3, 'label': 'v1', 'journal_closed': False}]


Step 3：在服务终端按 Ctrl+C，等待 shutdown、closed=True 和服务退出提示。

关闭客户端只结束客户端连接；还要关闭 Uvicorn。下一节沿用同一端口，须等当前服务退出后再启动。

## 4 用两个服务进程对照独立状态

每个服务进程都有自己的 count 和临时文件；一个进程的自增不会修改另一个进程的内存。先用官方图观察各 worker 的内存分别占用在哪里，再用两个服务进程对照状态归属。

访问第二个服务时，计数为什么重新从 1 开始？先把每个进程拥有的状态圈在自己的边界内，再追踪请求落到哪个端口。

![FastAPI 官方图：不同 worker 分别占用内存](image/illustration/15-01-process-state.svg)

引用原图：FastAPI Deployment Concepts，Copyright (c) 2018 Sebastián Ramírez，MIT 许可，图形内容未改动。Process Manager 是进程管理器，Worker Process 是工作进程，RAM 是内存；虚线箭头表示管理器启动 worker，图中的每个 1 GB 是官方举例，不是本实验的内存测量值。

本节不使用图中的管理进程，而是在 8150 与 8151 两个端口分别启动一个 worker，使请求固定落到某个进程。实验核对 PID、count 与临时文件的独立性，不验证管理器的请求分配，也不表示数据库或外部服务一定会被复制。

Step 1：在课程目录的终端启动 8150 上的单 worker 服务。

```bash
python -m uvicorn app:app --app-dir scripts/15-service-and-workers --host 127.0.0.1 --port 8150 --workers 1 --timeout-graceful-shutdown 5
```

Step 2：在课程目录的另一个终端启动 8151 上的单 worker 服务。

```bash
python -m uvicorn app:app --app-dir scripts/15-service-and-workers --host 127.0.0.1 --port 8151 --workers 1 --timeout-graceful-shutdown 5
```

Step 3：等待两个终端各自出现 Application startup complete，再运行下面的请求。

两个终端各运行一个独立服务，没有统一管理进程。本节验证进程的内存和资源归属；这与 --workers 2 在同一端口管理两名 worker 的启动方式不同，后者的机制在本节末尾说明。

按本节步骤启动两个服务，分别请求三次；核对 PID 不同、计数各自递增，而不是形成一条共享计数序列。

In [7]:
server_urls = ["http://127.0.0.1:8150", "http://127.0.0.1:8151"]
counts_by_pid = {}
for url in server_urls:
    with httpx.Client(base_url=url, timeout=2, trust_env=False) as client:
        results = []
        for _ in range(3):
            response = client.post("/tick")
            response.raise_for_status()
            results.append(response.json())
    pid = results[0]["pid"]
    assert all(item["pid"] == pid for item in results)
    assert all(item["journal_closed"] is False for item in results)
    counts_by_pid[pid] = [item["count"] for item in results]
assert len(counts_by_pid) == 2  # 两个端口应由不同服务进程处理。

将响应按 PID 分组，可以区分“各自从一开始累加”和“所有请求共用一个总数”。

In [8]:
for pid, counts in sorted(counts_by_pid.items()):
    assert counts == list(range(1, len(counts) + 1))
    assert len(counts) >= 3
    print(f"pid={pid}, counts={counts}")  # 预期：固定两组，每组 counts=[1, 2, 3]；PID 具体值及按 PID 排序后的先后可能变化。
# 两组计数都从 1 开始；不能只凭某一次 HTTP 200 判断双 worker 已就绪。

pid=30580, counts=[1, 2, 3]
pid=30676, counts=[1, 2, 3]


Uvicorn 的 --workers 2 会额外启动管理进程，由它启动和监控两个 worker，共用一个监听端口；内置多进程方式使用 spawn，也支持 Windows。请求可能因连接复用和调度连续落到同一 worker，不能要求严格轮流或均匀分配。

本机 Windows / Uvicorn 0.53.0 的共享端口启动仍会出现 WinError 10022；因此不能把下面的独立端口实验标记为共享端口部署通过。管理进程补起新 worker 后能够响应，不等于这次启动没有错误，因此当前实验使用前面的两个独立服务。这次观察不能替代共享端口多 worker 部署的验证。

多个 worker 仍独立持有内存并分别执行 lifespan。数据库连接池或大对象也要按进程数量考虑总资源占用；需要统一计数时，应使用所有进程共用的数据存储。

Uvicorn worker 运行整份 Web 应用并接收请求，ProcessPoolExecutor 则执行提交的 Python 函数。增加 worker 不会把一条路由中的计算自动拆分，也不保证吞吐量按数量等比增加。

Step 4：分别在两个服务终端按 Ctrl+C，核对各自出现 shutdown、closed=True，并等待两个进程退出。

## 5 通过手动重启更新源码

--reload 监听源码变化并重新启动服务进程，适合本地开发；正式运行使用明确的 worker 数量，并关闭开发重载。--reload 与 --workers 互斥，不同时传入。

当前 Windows / Uvicorn 0.53.0 的自动重载未通过进程替换检查。本节采用正常关闭再启动，分别观察新 PID、新源码标记和重置的内存状态；执行时沿用下面完整流程。

Step 1：在课程目录的终端重新启动一个 worker。

```bash
python -m uvicorn app:app --app-dir scripts/15-service-and-workers --host 127.0.0.1 --port 8150 --workers 1 --timeout-graceful-shutdown 5
```

Step 2：等待服务启动完成，记录更改前的响应。

In [9]:
with httpx.Client(base_url=BASE_URL, timeout=2, trust_env=False) as client:
    response = client.post("/tick")
    response.raise_for_status()
    before_restart = response.json()
assert before_restart["label"] == "v1" and before_restart["count"] == 1
print(before_restart)  # 预期：重启前 count=1、label='v1'、journal_closed=False；pid 为当前进程号。

{'pid': 17332, 'count': 1, 'label': 'v1', 'journal_closed': False}


Step 3：在服务终端按 Ctrl+C，等待 shutdown、closed=True 和进程退出。

Step 4：编辑配套 app.py，把 LABEL 的值从 "v1" 改成 "v2" 并保存。

Step 5：在课程目录的终端再次启动单 worker。

```bash
python -m uvicorn app:app --app-dir scripts/15-service-and-workers --host 127.0.0.1 --port 8150 --workers 1 --timeout-graceful-shutdown 5
```

Step 6：等待新 PID 的启动完成日志，再发出下面的请求。

In [10]:
with httpx.Client(base_url=BASE_URL, timeout=2, trust_env=False) as client:
    response = client.post("/tick")
    response.raise_for_status()
    after_restart = response.json()
assert after_restart["label"] == "v2"
assert after_restart["pid"] != before_restart["pid"]
assert after_restart["count"] == 1
print(after_restart)  # 预期：重启后 count 重新为 1、label='v2'、journal_closed=False，pid 改变。
# 新 PID、新标记和重新开始的计数，分别观察进程、源码与内存变化。

{'pid': 4440, 'count': 1, 'label': 'v2', 'journal_closed': False}


Step 7：在服务终端按 Ctrl+C，等待服务进程退出。

Step 8：把 app.py 中的 LABEL 恢复为 "v1" 并保存。

### 5.1 自动重载的机制与环境边界

--reload 用于监听源码变化并重新启动服务进程；--reload-dir 指定观察目录。没有 watchfiles 时，Uvicorn 也会轮询 Python 文件的修改时间。检测到变化只是第一步，旧 worker 正常退出、新 worker 启动并返回新标记，才表示替换完成。

当前 Windows / Uvicorn 0.53.0 的自动化启动环境仍观察到“检测到变化，但旧进程继续响应”，独立隐藏控制台的尝试也没有完成进程替换。因此，本章当前的可执行源码更新实验采用手动重启；自动重载的启动命令不列为操作练习。这个现象只说明当前环境的验证边界，不代表其他 Windows 环境都无法使用重载。

Uvicorn 的 Windows 重载实现涉及控制台退出信号。排查时需区分文件监视是否检测到变化、旧进程是否收到退出请求、以及新进程是否就绪；仅增加文件监视库不能证明进程替换问题已解决。

## 6 读日志、限制并发并确认退出

配套应用使用 uvicorn.error 记录 startup、shutdown 和资源关闭；这个记录器也接收普通服务信息。uvicorn.access 记录访问方法、路径和状态码。按 PID 对照生命周期日志，才能知道每个 worker 是否完成清理。

常用选项分别控制不同事情，示例数值只用于本地观察。

| 选项 | 中文名称／含义 | 边界 |
| --- | --- | --- |
| --log-level info | 日志级别 | 控制记录详细程度，不改变业务状态码 |
| --limit-concurrency | 并发连接或任务限制 | 达到限制时立即返回 503，不排队等待 |
| --timeout-graceful-shutdown 5 | 优雅关闭的请求等待时间 | 单位为秒；超时后开始终止未完成请求，不是任意清理代码都能保证完成的期限 |

并发限制还会计入空闲的持久连接。在本章 Uvicorn 0.53.0 的 h11 实现中，当前连接也计入检查；因此设为 1 会拒绝每个请求，不能把它理解为“同时放行一个请求”。用这个小反例观察 503。

Step 1：在课程目录的终端启动并发限制反例。

```bash
python -m uvicorn app:app --app-dir scripts/15-service-and-workers --host 127.0.0.1 --port 8150 --workers 1 --http h11 --limit-concurrency 1 --timeout-graceful-shutdown 5
```

Step 2：等待启动完成，再请求计数接口。

In [11]:
response = httpx.post(f"{BASE_URL}/tick", timeout=2, trust_env=False)
assert response.status_code == 503
print(response.status_code, response.text.strip())  # 预期：503 Service Unavailable。
# 请求被服务层拒绝，尚未执行 /tick；不是 FastAPI 参数校验错误。

503 Service Unavailable


Step 3：在服务终端按 Ctrl+C，等待资源关闭日志和进程退出。

优雅关闭会停止接受新连接，并等待正在进行的响应与任务，受所配置的等待时间约束；正常关闭随后执行 lifespan 的退出逻辑。检查终端已返回命令提示符，再确认本章地址无法连接。若只是关闭 HTTPX，服务仍然会运行。

In [12]:
for url in (BASE_URL, "http://127.0.0.1:8151"):
    try:
        httpx.get(f"{url}/state", timeout=1, trust_env=False)
    except (httpx.ConnectError, httpx.ConnectTimeout):
        print(url, "已无法连接")  # 预期：8150、8151 两个地址分别显示已无法连接。
    else:
        raise AssertionError(f"{url} 仍有服务响应，请检查服务终端")
# 未建立连接还须结合各服务的进程退出日志，不能只关闭客户端。

http://127.0.0.1:8150 已无法连接


http://127.0.0.1:8151 已无法连接


## 7 反向代理与 HTTPS 的职责

正式部署还要安排服务开机启动、崩溃后恢复、日志收集，以及 HTTPS 证书与续期。反向代理可以接收外部请求再转交给 Uvicorn；HTTPS 常由代理终止 TLS，再转发到应用服务器。本章本机 HTTP 实验不需要配置这些组件。

经过代理时，X-Forwarded-For 与 X-Forwarded-Proto 可传递原始客户端地址和协议。Uvicorn 的 --forwarded-allow-ips 指定允许提供这些信息的代理地址，只填写实际受信任的代理；任意外部客户端传来的同名头不能自动当作事实。"*" 表示信任所有来源，不作为未知网络环境的默认配置。

## 本章小结

（1）worker 运行完整应用；多个 worker 独立持有内存状态，并各自执行 lifespan 和资源清理。

（2）用 PID、每个 PID 的计数与生命周期日志共同观察服务。两个独立端口便于直接对照；--workers 则由管理进程组织共享端口的 worker。

（3）开发重载与多 worker 分开使用。当前环境以手动重启完成源码变更实验；新应用的进程内计数随初始化重置。

（4）日志、并发限制、优雅关闭和代理信任配置各有职责；服务退出还须核对进程与监听地址。

## 练习

1. 在单 worker 模式下完成三次 /tick，再正常关闭并重新启动。检查第二轮第一次调用的 count 为 1，且生命周期资源重新创建。

2. 在 8150、8151 各启动一个服务，只向 8150 连续发送三次 /tick，再分别请求两个端口的 /state。检查计数分别为 3、0，PID 不同，最后正常关闭两个服务。

3. 把并发限制反例的数值从 1 改为 10，重新启动服务并顺序调用 /tick。检查返回 200、计数正常增长；最后确认 worker 资源已关闭、8150 无服务响应。

提示：

（1）每轮从新启动的服务开始，只让本轮代码调用 /tick。

（2）第 2 题从两个全新服务开始；/state 不会增加计数，也没有共享内存计数器。

（3）第 3 题只检查少量顺序请求，不把结果当作压力测试或容量结论。

## 参考与引用来源

1. **FastAPI 官方文档**：[Server Workers](https://fastapi.tiangolo.com/deployment/server-workers/)，管理进程与 worker；[Deployment Concepts](https://fastapi.tiangolo.com/deployment/concepts/#memory-per-process) 的 Memory per Process、Previous Steps Before Starting 与 Security - HTTPS，进程内存、资源占用和部署职责；[Lifespan Events](https://fastapi.tiangolo.com/advanced/events/#lifespan) 与 [Testing Events](https://fastapi.tiangolo.com/advanced/testing-events/)，初始化、释放和 TestClient 上下文；Deployment Concepts 的 [Memory per Process 原图](https://fastapi.tiangolo.com/img/deployment/concepts/process-ram.drawio.svg)，各 worker 独立占用内存。

2. **Uvicorn 官方文档**：[Settings](https://uvicorn.dev/settings/) 的 Application、Socket Binding、Development、Production、Resource Limits、Timeouts 与 HTTP，命令参数及代理头信任；[Deployment](https://uvicorn.dev/deployment/#built-in)，spawn、进程管理与 Windows 支持；[Lifespan](https://uvicorn.dev/concepts/lifespan/)，每个 worker 独立执行生命周期；[Server Behavior](https://uvicorn.dev/server-behavior/#concurrency-and-backlog) 的 HTTP Headers、Concurrency and backlog 与 Graceful Process Shutdown，连接关闭、并发限制和优雅退出；[Logging](https://uvicorn.dev/concepts/logging/)，日志记录器和默认配置。

3. **Starlette 官方文档**：[Applications](https://starlette.dev/applications/#storing-state-on-the-app-instance) 的 Storing state on the app instance 与 Accessing the app instance，app.state 与 request.app。

4. **Python 3.12 官方文档**：[os.getpid](https://docs.python.org/3.12/library/os.html#os.getpid)，当前进程标识；[TemporaryFile](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryFile)，临时文件和自动关闭删除；[ProcessPoolExecutor](https://docs.python.org/3.12/library/concurrent.futures.html#processpoolexecutor)，计算任务进程池的职责。

5. **HTTPX 官方文档**：[Clients](https://www.python-httpx.org/advanced/clients/)，连接复用与上下文清理；[Timeouts](https://www.python-httpx.org/advanced/timeouts/)，等待超时条件；[Environment Variables](https://www.python-httpx.org/environment_variables/)，trust_env；[Exceptions](https://www.python-httpx.org/exceptions/)，ConnectError 与 ConnectTimeout。

6. **Uvicorn 官方源码**：[0.53.0 的 h11_impl.py](https://raw.githubusercontent.com/Kludex/uvicorn/0.53.0/uvicorn/protocols/http/h11_impl.py)，H11Protocol.connection_made 将当前连接加入连接集合，handle_events 用连接数或任务数大于等于 limit_concurrency 的条件选择 service_unavailable；直接支持本章数值为 1 的反例；[0.53.0 的 basereload.py](https://raw.githubusercontent.com/Kludex/uvicorn/0.53.0/uvicorn/supervisors/basereload.py) 的 BaseReload.restart，支持 Windows 控制台信号、等待旧进程退出与启动替代进程的机制说明。同一源码托管站点的 [FastAPI 图片源文件](https://github.com/fastapi/fastapi/blob/master/docs/en/docs/img/deployment/concepts/process-ram.drawio.svg)及[仓库 MIT 许可](https://github.com/fastapi/fastapi/blob/master/LICENSE)，Copyright (c) 2018 Sebastián Ramírez；所引图片保留 MIT 许可全文于 SVG 元数据，图形内容未改动。